# MNIST Rotation Attacks With a Simple Neural Network

This notebook studies empirical rotation attacks on a simple MLP.

This is not certification. The attacker searches a finite grid of rotation angles and succeeds if any tested angle changes a previously correct prediction.

In [ ]:
import gzip
import random
import struct
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = Path("mnist")
BASELINE_MODEL_PATH = Path("mnist_rotation_baseline.pt")
AUGMENTED_MODEL_PATH = Path("mnist_rotation_augmented.pt")

RESET_MODELS = True  # set False after one full run to reuse saved weights

TRAIN_LIMIT = 12000
TEST_LIMIT = 1000
ATTACK_LIMIT = 500

BASELINE_EPOCHS = 5
AUGMENTED_EPOCHS = 5
BATCH_SIZE = 128
LR = 1e-3

MAX_TRAIN_ROTATION_DEG = 30
ATTACK_ANGLES = list(range(-45, 46, 5))
NONZERO_ATTACK_ANGLES = [a for a in ATTACK_ANGLES if a != 0]

print(f"device={DEVICE}")

In [ ]:
BASE_URL = "https://ossci-datasets.s3.amazonaws.com/mnist"
MNIST_FILES = [
    "train-images-idx3-ubyte.gz",
    "train-labels-idx1-ubyte.gz",
    "t10k-images-idx3-ubyte.gz",
    "t10k-labels-idx1-ubyte.gz",
]


def ensure_mnist(data_dir=DATA_DIR):
    data_dir.mkdir(exist_ok=True)
    for filename in MNIST_FILES:
        raw_path = data_dir / filename[:-3]
        gz_path = data_dir / filename
        if raw_path.exists():
            continue

        print(f"downloading {filename}")
        urllib.request.urlretrieve(f"{BASE_URL}/{filename}", gz_path)
        with gzip.open(gz_path, "rb") as src, open(raw_path, "wb") as dst:
            dst.write(src.read())
        gz_path.unlink()


def read_images(path):
    with open(path, "rb") as f:
        magic, n, rows, cols = struct.unpack(">IIII", f.read(16))
        if magic != 2051:
            raise ValueError(f"bad image file: {path}")
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(n, rows * cols).astype(np.float32) / 255.0


def read_labels(path):
    with open(path, "rb") as f:
        magic, n = struct.unpack(">II", f.read(8))
        if magic != 2049:
            raise ValueError(f"bad label file: {path}")
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.astype(np.int64)


def load_mnist(data_dir=DATA_DIR):
    ensure_mnist(data_dir)
    x_train = read_images(data_dir / "train-images-idx3-ubyte")
    y_train = read_labels(data_dir / "train-labels-idx1-ubyte")
    x_test = read_images(data_dir / "t10k-images-idx3-ubyte")
    y_test = read_labels(data_dir / "t10k-labels-idx1-ubyte")

    x_train = torch.from_numpy(x_train).to(DEVICE)
    y_train = torch.from_numpy(y_train).to(DEVICE)
    x_test = torch.from_numpy(x_test).to(DEVICE)
    y_test = torch.from_numpy(y_test).to(DEVICE)
    return x_train, y_train, x_test, y_test


X_train, y_train, X_test, y_test = load_mnist()
print(f"train={tuple(X_train.shape)}  test={tuple(X_test.shape)}")

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.layers(x)


def batched_indices(n, batch_size, shuffle=True):
    if shuffle:
        order = torch.randperm(n, device=DEVICE)
    else:
        order = torch.arange(n, device=DEVICE)
    for start in range(0, n, batch_size):
        yield order[start:start + batch_size]


@torch.no_grad()
def accuracy(model, x, y, limit=None, batch_size=512):
    model.eval()
    if limit is not None:
        x = x[:limit]
        y = y[:limit]

    correct = 0
    total = 0
    for idx in batched_indices(len(x), batch_size, shuffle=False):
        pred = model(x[idx]).argmax(dim=1)
        correct += (pred == y[idx]).sum().item()
        total += len(idx)
    return correct / total


def train_clean(model, x, y, epochs, batch_size, lr, limit):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    x = x[:limit]
    y = y[:limit]

    for epoch in range(1, epochs + 1):
        losses = []
        model.train()
        for idx in batched_indices(len(x), batch_size, shuffle=True):
            logits = model(x[idx])
            loss = F.cross_entropy(logits, y[idx])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            losses.append(loss.item())

        acc = accuracy(model, X_test, y_test, limit=TEST_LIMIT)
        print(f"clean epoch {epoch:02d}/{epochs}  loss={np.mean(losses):.4f}  test_acc={100 * acc:.2f}%")


def save_model(model, path):
    torch.save(model.state_dict(), path)


def load_model(path):
    model = MLP().to(DEVICE)
    state = torch.load(path, map_location=DEVICE)
    model.load_state_dict(state)
    return model

In [ ]:
def rotate_batch(x, angle_degrees):
    images = x.view(-1, 1, 28, 28)
    batch = images.shape[0]

    if not torch.is_tensor(angle_degrees):
        angle_degrees = torch.full((batch,), float(angle_degrees), device=x.device)
    else:
        angle_degrees = angle_degrees.to(x.device).float()
        if angle_degrees.ndim == 0:
            angle_degrees = angle_degrees.repeat(batch)

    theta_rad = angle_degrees * torch.pi / 180.0
    cos = torch.cos(theta_rad)
    sin = torch.sin(theta_rad)

    theta = torch.zeros(batch, 2, 3, device=x.device, dtype=x.dtype)
    theta[:, 0, 0] = cos
    theta[:, 0, 1] = -sin
    theta[:, 1, 0] = sin
    theta[:, 1, 1] = cos

    grid = F.affine_grid(theta, images.shape, align_corners=False)
    rotated = F.grid_sample(
        images,
        grid,
        mode="bilinear",
        padding_mode="zeros",
        align_corners=False,
    )
    return rotated.view(batch, 784)


def random_rotate_batch(x, max_abs_degrees):
    angles = torch.empty(len(x), device=x.device).uniform_(-max_abs_degrees, max_abs_degrees)
    return rotate_batch(x, angles)

In [ ]:
@torch.no_grad()
def rotation_accuracy_by_angle(model, x, y, angles, limit=ATTACK_LIMIT, batch_size=256):
    model.eval()
    x = x[:limit]
    y = y[:limit]
    rows = []

    for angle in angles:
        correct = 0
        total = 0
        for idx in batched_indices(len(x), batch_size, shuffle=False):
            xb = rotate_batch(x[idx], angle)
            pred = model(xb).argmax(dim=1)
            correct += (pred == y[idx]).sum().item()
            total += len(idx)
        rows.append({"angle_deg": angle, "accuracy": correct / total})

    return pd.DataFrame(rows)


@torch.no_grad()
def rotation_attack_report(model, x, y, angles, limit=ATTACK_LIMIT, batch_size=128):
    model.eval()
    x = x[:limit]
    y = y[:limit]

    clean_correct = 0
    grid_robust = 0
    attacked = 0

    for idx in batched_indices(len(x), batch_size, shuffle=False):
        xb = x[idx]
        yb = y[idx]

        clean_pred = model(xb).argmax(dim=1)
        clean_ok = clean_pred == yb
        any_failed = torch.zeros_like(clean_ok)

        for angle in angles:
            rotated = rotate_batch(xb, angle)
            pred = model(rotated).argmax(dim=1)
            any_failed |= pred != yb

        clean_correct += clean_ok.sum().item()
        attacked += (clean_ok & any_failed).sum().item()
        grid_robust += (clean_ok & ~any_failed).sum().item()

    return pd.DataFrame([
        {
            "clean_acc": clean_correct / len(x),
            "grid_rotation_robust_acc": grid_robust / len(x),
            "attack_success_on_clean_correct": attacked / max(1, clean_correct),
            "angles_tested": len(angles),
            "min_angle": min(angles),
            "max_angle": max(angles),
        }
    ])


@torch.no_grad()
def first_rotation_failures(model, x, y, angles, limit=100, max_rows=20):
    model.eval()
    rows = []
    x = x[:limit]
    y = y[:limit]

    for i in range(len(x)):
        xi = x[i:i + 1]
        yi = int(y[i].item())
        clean_pred = int(model(xi).argmax(dim=1).item())
        if clean_pred != yi:
            continue

        for angle in angles:
            pred = int(model(rotate_batch(xi, angle)).argmax(dim=1).item())
            if pred != yi:
                rows.append({
                    "image_index": i,
                    "true_label": yi,
                    "clean_pred": clean_pred,
                    "first_bad_angle": angle,
                    "rotated_pred": pred,
                })
                break

        if len(rows) >= max_rows:
            break

    return pd.DataFrame(rows)

## 1. Baseline Model

Train a normal MLP and evaluate how often a grid-search rotation attack can change correct predictions.

In [ ]:
if BASELINE_MODEL_PATH.exists() and not RESET_MODELS:
    baseline_model = load_model(BASELINE_MODEL_PATH)
else:
    baseline_model = MLP().to(DEVICE)
    train_clean(baseline_model, X_train, y_train, BASELINE_EPOCHS, BATCH_SIZE, LR, TRAIN_LIMIT)
    save_model(baseline_model, BASELINE_MODEL_PATH)

print(f"baseline_clean_acc={100 * accuracy(baseline_model, X_test, y_test, limit=TEST_LIMIT):.2f}%")

display(rotation_attack_report(
    baseline_model,
    X_test,
    y_test,
    NONZERO_ATTACK_ANGLES,
).style.format({
    "clean_acc": "{:.2%}",
    "grid_rotation_robust_acc": "{:.2%}",
    "attack_success_on_clean_correct": "{:.2%}",
}))

display(rotation_accuracy_by_angle(
    baseline_model,
    X_test,
    y_test,
    ATTACK_ANGLES,
).style.format({"accuracy": "{:.2%}"}))

## 2. Rotation-Augmented Training

This is an empirical defense. During training, each batch is mixed with randomly rotated versions of the same images.

In [ ]:
def train_rotation_augmented(model, x, y, epochs, batch_size, lr, limit, max_abs_degrees):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    x = x[:limit]
    y = y[:limit]

    for epoch in range(1, epochs + 1):
        losses = []
        model.train()
        for idx in batched_indices(len(x), batch_size, shuffle=True):
            xb = x[idx]
            yb = y[idx]
            xb_rot = random_rotate_batch(xb, max_abs_degrees)

            train_x = torch.cat([xb, xb_rot], dim=0)
            train_y = torch.cat([yb, yb], dim=0)
            logits = model(train_x)
            loss = F.cross_entropy(logits, train_y)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            losses.append(loss.item())

        acc = accuracy(model, X_test, y_test, limit=TEST_LIMIT)
        print(f"rotation epoch {epoch:02d}/{epochs}  loss={np.mean(losses):.4f}  clean_acc={100 * acc:.2f}%")


if AUGMENTED_MODEL_PATH.exists() and not RESET_MODELS:
    augmented_model = load_model(AUGMENTED_MODEL_PATH)
else:
    augmented_model = MLP().to(DEVICE)
    augmented_model.load_state_dict(baseline_model.state_dict())
    train_rotation_augmented(
        augmented_model,
        X_train,
        y_train,
        AUGMENTED_EPOCHS,
        BATCH_SIZE,
        LR,
        TRAIN_LIMIT,
        MAX_TRAIN_ROTATION_DEG,
    )
    save_model(augmented_model, AUGMENTED_MODEL_PATH)

print(f"rotation_augmented_clean_acc={100 * accuracy(augmented_model, X_test, y_test, limit=TEST_LIMIT):.2f}%")

display(rotation_attack_report(
    augmented_model,
    X_test,
    y_test,
    NONZERO_ATTACK_ANGLES,
).style.format({
    "clean_acc": "{:.2%}",
    "grid_rotation_robust_acc": "{:.2%}",
    "attack_success_on_clean_correct": "{:.2%}",
}))

display(rotation_accuracy_by_angle(
    augmented_model,
    X_test,
    y_test,
    ATTACK_ANGLES,
).style.format({"accuracy": "{:.2%}"}))

## 3. Final Comparison

The main metric is `grid_rotation_robust_acc`: the fraction of tested images that are clean-correct and stay correct for every tested nonzero rotation angle.

In [ ]:
def final_rotation_summary(models):
    rows = []
    for name, model in models:
        report = rotation_attack_report(
            model,
            X_test,
            y_test,
            NONZERO_ATTACK_ANGLES,
            limit=ATTACK_LIMIT,
        ).iloc[0]
        rows.append({
            "model": name,
            "clean_acc": report["clean_acc"],
            "grid_rotation_robust_acc": report["grid_rotation_robust_acc"],
            "attack_success_on_clean_correct": report["attack_success_on_clean_correct"],
            "angles_tested": int(report["angles_tested"]),
            "angle_range": f"{int(report['min_angle'])}..{int(report['max_angle'])}",
        })
    return pd.DataFrame(rows)


summary = final_rotation_summary([
    ("baseline", baseline_model),
    ("rotation_augmented", augmented_model),
])

display(summary.style.format({
    "clean_acc": "{:.2%}",
    "grid_rotation_robust_acc": "{:.2%}",
    "attack_success_on_clean_correct": "{:.2%}",
}))

## 4. First Rotation Failures

This table lists examples where the baseline model is correct on the clean image but fails for at least one tested rotation angle.

In [ ]:
display(first_rotation_failures(
    baseline_model,
    X_test,
    y_test,
    NONZERO_ATTACK_ANGLES,
    limit=ATTACK_LIMIT,
    max_rows=20,
))